# 03. Attention của ViT-B/16

Lấy attention **CLS-to-patch**, trung bình qua 12 head, tại block **3/6/9/12**.
Giữ ViT-B/16 IMAGENET1K_V1 và preprocessing chính thức của baseline.
Entropy mô tả phân bố không gian, không chứng minh khả năng định vị đối tượng hoặc quan hệ nhân quả.
Attention không phụ thuộc lớp đích. Phân tích cả lưới 7×7/14×14; bảng và hình chính dùng 14×14.

Notebook chạy độc lập: setup → chọn ảnh → chạy/resume → phân tích → export.
`experiments/attention/`: `methods.py` (heatmap), `experiment.py` (cohort/checkpoint),
`analysis.py` (metrics/bảng/hình/export). Tái sử dụng `src` và giữ nguyên các module chung.

## 1. Environment and synchronization

Colab: open this notebook from the personal public repository and select a GPU runtime.
Run the setup cell to clone/pull `feat/gradcam`, mount Drive, prepare data and install dependencies.
No GitHub token is needed. On Windows, run `.\sync.ps1` after editing, then rerun setup here.

Input: `MyDrive/cv_2026/data_10k.zip` containing 14 Parquet shards under `data_10k/`.
Setup extracts the shards to `/content` once per runtime and verifies the manifest and IDs/labels.
Code always comes from Git; set `DATA_DIR` only to override the extracted dataset.
Results/checkpoints stay under `MyDrive/cv_2026/metrics/` and `MyDrive/cv_2026/figures/`.
Weights are cached under `MyDrive/cv_2026/torch-cache/`.

Stop an active run before pulling. When available, autoreload refreshes Python modules before
subsequent cells; rerun configuration and experiment cells to recreate objects. If the runtime
cannot load autoreload, setup continues; restart the Python session after code changes. Restart the runtime session when
changing model classes or dependencies, or if an old object still behaves unexpectedly.
Git pull updates Python files, not the notebook cells already open in the Colab editor.
Reopen this notebook from GitHub when its cells change.

Local: use the local repository and dataset; no Drive mount or automatic dependency installation.
See [workflow guide](../docs/personal_remote_colab.md).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import runpy

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab defaults are ready to use. Local execution keeps the local repository.
PROJECT_DIR = None
DATA_DIR = None
RESULTS_DIR = None
PROJECT_ZIP = "cv_2026/data_10k.zip"  # Input ZIP relative to MyDrive.
MOUNT_DRIVE = True
INSTALL_DEPENDENCIES = IN_COLAB
REPO_URL = "https://github.com/thuhang012/cnn-vit-interpretability-personal.git"
BRANCH = "feat/gradcam"

if IN_COLAB:
    checkout = Path(PROJECT_DIR or "/content/cnn-vit-interpretability").resolve()

    def git(*args):
        return subprocess.check_output(["git", *args], text=True).strip()

    if (checkout / ".git").is_dir():
        if git("-C", str(checkout), "remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("Checkout points to a different repository.")
        if git("-C", str(checkout), "branch", "--show-current") != BRANCH:
            raise RuntimeError("Checkout is on a different branch.")
        if git("-C", str(checkout), "status", "--porcelain"):
            raise RuntimeError("Save Colab checkout edits separately before pulling.")
        git("-C", str(checkout), "pull", "--ff-only", "origin", BRANCH)
    else:
        git("clone", "--branch", BRANCH, REPO_URL, str(checkout))
    print("Code commit:", git("-C", str(checkout), "rev-parse", "--short", "HEAD"))
    configure = runpy.run_path(str(checkout / "scripts/colab_setup.py"))["setup"]
    ROOT, DATA_DIR, RESULTS_DIR = configure(
        checkout, data_dir=DATA_DIR, results_dir=RESULTS_DIR,
        project_zip=PROJECT_ZIP, mount_drive=MOUNT_DRIVE,
        install_dependencies=INSTALL_DEPENDENCIES,
    )
else:
    default_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    ROOT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else default_root.resolve()

    def project_path(value, default):
        path = Path(value).expanduser() if value is not None else ROOT / default
        return (path if path.is_absolute() else ROOT / path).resolve()

    DATA_DIR = project_path(DATA_DIR, "data/imagenet/data")
    RESULTS_DIR = project_path(RESULTS_DIR, "results")
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                        str(ROOT / "requirements.txt")], check=True)

if not (ROOT / "configs/analysis_set.json").is_file():
    raise FileNotFoundError("PROJECT_DIR must point to the repository root.")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder is missing: {DATA_DIR}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload edited Python modules on subsequent cells after pulling new code.
try:
    shell = get_ipython()
except NameError:
    shell = None
AUTORELOAD_AVAILABLE = False
if shell is not None:
    try:
        shell.run_line_magic("load_ext", "autoreload")
        shell.run_line_magic("autoreload", "2")
        AUTORELOAD_AVAILABLE = True
    except ImportError as error:
        print(f"Optional autoreload is unavailable ({error}). Setup can continue.")
        print("After pulling Python code changes, restart the Python session and rerun cells to load the new code.")
print(f"Environment: {'Colab' if IN_COLAB else 'local'} | Project: {ROOT}")
print(f"Data: {DATA_DIR}\nResults/checkpoints: {RESULTS_DIR}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from experiments.attention.experiment import prepare_cohorts, experiment_stores
from experiments.attention.analysis import analyze, show_results, plot_summary, plot_examples, export_final

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

In [ ]:
RUN_EXPERIMENT = True      # False: only read existing checkpoints.
SAVE_FINAL = False         # Preview until results have been reviewed.
CHECK_PINNED_DATA = False  # Optional full source/hash verification.
ANALYSIS_SIZE = 10000      # 5000: class-balanced trial; 10000: shared cohort.
SEED = 42
BATCH_SIZE = 1             # Explicit batch; no pilot or automatic batch selection.
ALLOW_DOWNLOAD = IN_COLAB    # Opt in if pretrained weights are missing.
RUN_TAG = f"attention_v1_n{ANALYSIS_SIZE}"
CONDITION = "E2"

METRICS_ROOT = RESULTS_DIR / "metrics/experiments/attention" / RUN_TAG
OUT = METRICS_ROOT / "intermediate"
FINAL = METRICS_ROOT / "final"
FIG = RESULTS_DIR / "figures/experiments/attention" / RUN_TAG / "final"
stores = result = None
figures = {}

## 2. Manifest và cohort

Giữ nguyên manifest 10.000 ảnh (10 ảnh/lớp), seed và cách ghép dự đoán baseline theo ID.
Nhóm đúng/sai vẫn dựa trên baseline của cả ResNet và ViT; không cần chạy lại baseline.
Attention dùng một ảnh/lớp (1.000 ảnh), chọn bằng seed 42 từ cohort đã chọn.
`ANALYSIS_SIZE` vẫn xác định cohort nguồn như notebook cũ; không đổi thành số ảnh attention.

In [ ]:
if CHECK_PINNED_DATA:
    # The existing checker uses the dataset path in configs/analysis_set.json.
    if DATA_DIR != (ROOT / "data/imagenet/data").resolve():
        raise ValueError("CHECK_PINNED_DATA uses the configured dataset; run it with the default DATA_DIR.")
    subprocess.run([sys.executable, str(ROOT / "src/create_analysis_manifest.py"), "--check"],
                   cwd=ROOT, check=True)
selection = prepare_cohorts(ROOT, ANALYSIS_SIZE, SEED)
display(selection.groupby("group").size().rename("n_images").to_frame())
print(f"{CONDITION}: {len(selection)} images | seed={SEED}")

## 3. Chạy hoặc tiếp tục checkpoint

Không yêu cầu pilot. Batch mặc định 1; tăng `BATCH_SIZE` khi phù hợp bộ nhớ.
Các kiểm tra logits, heatmap và hook vẫn được giữ trong lúc trích xuất.
ID đã commit không chạy lại; checkpoint khác cấu hình bị từ chối.
OOM dừng lượt chạy, không tự giảm cohort. Chọn run ID mới khi thay code/cấu hình/batch.
Để xem lại, đặt `RUN_EXPERIMENT=False` và `SAVE_FINAL=False`, giữ cùng run ID.

Completed compatible attention checkpoints are read automatically before inference.
Notebook-only commits do not force recomputation; original provenance is preserved.
Run all reuses saved maps and skips final export by default. Incomplete runs keep strict resume checks.


In [ ]:
if not isinstance(BATCH_SIZE, int) or isinstance(BATCH_SIZE, bool) or BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be a positive integer")

# Read completed checkpoints before considering inference. A notebook-only commit
# changes git_revision but does not change the attention maps already computed.
from experiments.attention.experiment import RunStore, checksum
import hashlib

cached_stores = []
expected_ids = set(selection.image_id)
expected_records = hashlib.sha256(selection.sort_values("image_id").to_csv(index=False).encode()).hexdigest()
expected_config = {
    "experiment": CONDITION, "seed": SEED, "records": expected_records,
    "target": "class_agnostic", "preprocessing": "official",
    "grids": [7, 14], "top_fraction": 0.2,
    "manifest": checksum(ROOT / "data/manifests/analysis_10000_seed42.csv"),
    "dataset_config": checksum(ROOT / "configs/analysis_set.json"),
    "code": {name: checksum(ROOT / "experiments/attention" / name)
             for name in ("methods.py", "experiment.py", "analysis.py")},
    "shared_code": {name: checksum(ROOT / "src" / name)
                    for name in ("models.py", "data.py", "evaluation.py", "evaluate_baseline.py")},
    "baseline": {name: checksum(ROOT / f"results/metrics/baseline_notebook/full/{name}_predictions.csv")
                 for name in ("resnet50", "vit_b_16")},
}
for model_name in ("vit_b_16",):
    checkpoint_dir = OUT / CONDITION / model_name
    cached = RunStore(checkpoint_dir) if (checkpoint_dir / "config.json").is_file() else None
    if cached is not None:
        mismatches = [key for key, value in {**expected_config, "model": model_name}.items()
                      if cached.config.get(key) != value]
        if set(cached.config["image_ids"]) != expected_ids:
            mismatches.append("image_ids")
        if mismatches:
            raise ValueError(f"{model_name}: checkpoint differs in {', '.join(mismatches)}. "
                             "Keep the existing run unchanged; choose a new RUN_TAG for a different experiment.")
    cached_stores.append(cached)

if all(store is not None and store.done == expected_ids for store in cached_stores):
    stores = cached_stores
    for store in stores:
        print(f"{CONDITION}/{store.config['model']}: reused {len(store.done)}/{len(expected_ids)} "
              f"saved attention maps; original commit={store.config.get('git_revision', 'unknown')}; no inference or writes")
else:
    batch_sizes = {name: BATCH_SIZE for name in ("vit_b_16",)} if RUN_EXPERIMENT else None
    try:
        stores = experiment_stores(ROOT, selection, CONDITION, OUT, batch_sizes=batch_sizes,
                                   device=DEVICE, seed=SEED, allow_download=ALLOW_DOWNLOAD,
                                   data_dir=DATA_DIR)
    except ValueError as error:
        if "Checkpoint configuration mismatch:" not in str(error):
            raise
        from experiments.attention.experiment import _identity
        for store in cached_stores:
            if store is None or store.done == expected_ids:
                continue
            current = _identity(ROOT, selection, store.config["model"], CONDITION,
                                BATCH_SIZE, torch.device(DEVICE), seed=SEED,
                                run_id=OUT.parent.name, model_metadata=store.config.get("model_metadata"),
                                data_dir=DATA_DIR)
            changed = sorted(key for key in set(current) | set(store.config)
                             if current.get(key) != store.config.get(key))
            print(f"Incomplete {CONDITION}/{store.config['model']}: {len(store.done)}/{len(expected_ids)} processed; "
                  f"changed configuration fields: {', '.join(changed)}")
        raise ValueError("Incomplete attention checkpoint cannot be resumed with this configuration. "
                         "Existing files are unchanged; use the changed-field report above to resolve it.") from error
result = analyze(*stores) if stores is not None else None


## 4. Chỉ số và độ bao phủ

Bảng tổng hợp giữ nguyên nội dung: chỉ số chính, nhóm đúng/sai và coverage.
Heatmap lỗi/suy biến được gắn cờ; metric không xác định giữ NaN, không thay bằng 0.
Không hiển thị bảng per-image hoặc danh sách ID đầy đủ; các bảng này có trong file export.

In [ ]:
if result is not None:
    show_results({CONDITION: result})
else:
    print(f"{CONDITION} chưa hoàn tất; xem tiến độ ở phần trên.")

## 5. Visualization

Giữ nguyên biểu đồ định lượng, cách chọn ảnh theo nhóm baseline và các panel heatmap.
Mỗi input/overlay dùng cùng crop chính thức; tiêu đề ghi rõ model và phương pháp.

In [ ]:
figures = {}
if result is not None:
    figures = {f"{CONDITION}_metrics": plot_summary(result, CONDITION),
               f"{CONDITION}_examples": plot_examples(ROOT, stores, selection, data_dir=DATA_DIR)}
    plt.show()

## 6. Export riêng cho E2

Không cần chạy notebook còn lại. Lưu `metrics.csv`, `per_image.csv`,
CSV dự đoán của ViT và `metadata.json` vào `FINAL`;
hai hình `E2_metrics.png`, `E2_examples.png` vào `FIG`.
Giữ schema số liệu cũ; tên thí nghiệm trong metadata/CSV là `attention`.

Accuracy dùng dự đoán trích xuất thành công, kèm số ảnh yêu cầu/lỗi; đây là kiểm tra
tính nhất quán dự đoán, không phải thước đo chất lượng heatmap.
Không ghi đè final hoặc chuyển checkpoint cũ từ notebook chung.
Trên Colab, checkpoint và final đều được lưu trên Drive; khi ngắt phiên có thể mất
phần batch chưa commit, nhưng lượt chạy sau tiếp tục từ các chunk đã lưu.
Notebook được lưu không kèm output nhúng.

In [ ]:
if SAVE_FINAL and result is not None:
    accuracy = export_final(stores, {CONDITION: result}, figures, FINAL, FIG)
    print("Prediction accuracy: percentages on successful extractions; failures listed separately")
    display(accuracy.round(4))
elif not SAVE_FINAL:
    print("Preview only: no final files written.")
else:
    print(f"Final export skipped: {CONDITION} is incomplete.")

## 7. RQ2 - How does a query patch attend to other patches?

This supplement preserves the completed CLS analysis above. It uses **direct query-to-key
attention** at blocks 3/6/9/12, averaged over 12 heads, not CLS maps or attention rollout.
The old checkpoints do not contain patch-query rows, so one additional forward per image
is required. No RQ2 checkpoints, figures or result files are saved; everything stays in memory.

**Distant Attention Mass (DAM).** On the native 14x14 grid, patch centers have integer
(row, column) coordinates. A destination is distant when Euclidean distance >= 7 patch
steps (112 input pixels). Exclude CLS and the query itself, then renormalize attention over
the 195 remaining patches. For query q:

`DAM(q) = sum(A[q, k] for distant k) / sum(A[q, k] for patch k != q)`.

The uniform baseline is `number of distant patches / 195`, calculated separately for each
query. This controls for boundary geometry and the large number of distant destinations.
Near mass is `1 - DAM`, for distance < 7 excluding self. Average queries within each image,
then images equally; report DAM, baseline and their paired difference in percentage points.
Report excluded CLS+self mass to make the conditional denominator explicit. A zero denominator
is undefined, not a zero DAM. Uniform maps are valid here and are not removed as degenerate.

Default: use the same E2 selection (currently one image per class, 1,000 images when all classes
are present), all 196 queries per image. No correctness filtering or attention-based selection.
For a smoke test only, set `RQ2_IMAGE_LIMIT = 5`; the output labels it as a preview.
Two deterministic example images and queries (6,6), (3,3) illustrate four blocks; visualization
color limits are shared across blocks for each query. Query cells are masked, not scored zero.

Interpretation: positive DAM demonstrates measured nonlocal attention allocation; DAM above
uniform indicates enrichment beyond spatial availability. Attention allocation alone does not
prove causal influence, object correspondence or a model-quality advantage. No entropy, IoU
or connected-components metrics are added to RQ2.


In [ ]:
import gc
from contextlib import contextmanager
import numpy as np
from src.models import load_model
from src.data import iter_analysis_images

RQ2_BLOCKS = (3, 6, 9, 12)
RQ2_GRID = 14
RQ2_FAR_DISTANCE = 7.0
RQ2_IMAGE_LIMIT = None  # None: full E2 cohort; 5: explicit smoke preview.
RQ2_QUERIES = ((6, 6), (3, 3))  # Zero-based row/column; selected before seeing attention.


def _rq2_geometry(grid, distance):
    if grid < 2 or not np.isfinite(distance) or distance <= 0:
        raise ValueError("Use a grid >= 2 and a positive finite distance.")
    coords = np.stack(np.unravel_index(np.arange(grid * grid), (grid, grid)), axis=1)
    squared = np.square(coords[:, None] - coords[None, :]).sum(-1)
    return squared >= distance ** 2


def _rq2_measure(weights, far):
    """Rows=query, columns=key. Average heads before calling; exclude CLS and self."""
    weights = np.asarray(weights, dtype=np.float64)
    n = len(far)
    if weights.shape != (n + 1, n + 1) or far.shape != (n, n):
        raise ValueError("Unexpected token/query/key dimensions.")
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Attention must be finite and nonnegative.")
    if not np.allclose(weights.sum(-1), 1, atol=1e-4, rtol=0):
        raise ValueError("Attention rows do not sum to one.")
    patch = weights[1:, 1:].copy()
    excluded = weights[1:, 0] + np.diag(patch)
    np.fill_diagonal(patch, 0)
    denominator = patch.sum(-1)
    conditional = np.divide(patch, denominator[:, None], out=np.full_like(patch, np.nan),
                            where=denominator[:, None] > 0)
    dam = np.sum(np.where(far, conditional, 0), axis=1)
    dam[denominator == 0] = np.nan
    return {"dam": dam, "uniform": far.sum(-1) / (n - 1),
            "excluded": excluded, "conditional": conditional}


@contextmanager
def _rq2_capture(model):
    """Scoped hooks: rows are query tokens; token zero is CLS. Always remove hooks."""
    handles, captured = [], {}

    def request(module, args, kwargs):
        return args, {**kwargs, "need_weights": True, "average_attn_weights": True}

    def collect(module, args, output, block):
        weights = output[1]
        tokens = RQ2_GRID ** 2 + 1
        if weights is None or tuple(weights.shape) != (1, tokens, tokens):
            raise ValueError(f"Unexpected head-averaged attention at block {block}.")
        captured[block] = weights[0].detach().float().cpu().numpy()

    try:
        for block in RQ2_BLOCKS:
            module = model.encoder.layers[block - 1].self_attention
            handles.append(module.register_forward_pre_hook(request, with_kwargs=True))
            handles.append(module.register_forward_hook(lambda m, a, o, b=block: collect(m, a, o, b)))
        yield captured
    finally:
        for handle in handles:
            handle.remove()


In [ ]:
# Additional inference only; completed E2 stores and result variables are not modified.
rq2_selection = selection.sort_values("image_id").copy()
if RQ2_IMAGE_LIMIT is not None:
    if isinstance(RQ2_IMAGE_LIMIT, bool) or not isinstance(RQ2_IMAGE_LIMIT, int) or RQ2_IMAGE_LIMIT < 1:
        raise ValueError("RQ2_IMAGE_LIMIT must be None or a positive integer.")
    rq2_selection = rq2_selection.head(RQ2_IMAGE_LIMIT)
if rq2_selection.empty or rq2_selection.image_id.duplicated().any():
    raise ValueError("RQ2 requires nonempty unique image IDs.")
if any(not (0 <= row < RQ2_GRID and 0 <= col < RQ2_GRID) for row, col in RQ2_QUERIES):
    raise ValueError("Query coordinates are outside the patch grid.")
rq2_far = _rq2_geometry(RQ2_GRID, RQ2_FAR_DISTANCE)
rq2_example_ids = set(rq2_selection.groupby("group", sort=True).head(1).head(2).image_id)
rq2_image_metrics, rq2_examples = [], []
rq2_model, rq2_transform, rq2_metadata = load_model("vit_b_16", DEVICE, allow_download=ALLOW_DOWNLOAD)
if rq2_model.image_size // rq2_model.patch_size != RQ2_GRID:
    raise ValueError("RQ2 requires the fixed ViT-B/16 14x14 patch grid.")
print(f"RQ2: {len(rq2_selection)} images, {RQ2_GRID ** 2} queries/image, blocks={RQ2_BLOCKS}; "
      f"far distance >= {RQ2_FAR_DISTANCE:g} patch steps; no result files written")
if len(rq2_selection) < len(selection):
    print("SMOKE PREVIEW ONLY: this is not the full E2 cohort.")
try:
    for rq2_count, (rq2_image, _, rq2_id) in enumerate(iter_analysis_images(
            rq2_selection.to_dict("records"), data_dir=DATA_DIR), 1):
        rq2_tensor = rq2_transform(rq2_image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            rq2_reference = rq2_model(rq2_tensor) if rq2_count == 1 else None
            with _rq2_capture(rq2_model) as rq2_weights:
                rq2_logits = rq2_model(rq2_tensor)
        if not torch.isfinite(rq2_logits).all():
            raise ValueError(f"Nonfinite logits for {rq2_id}.")
        if rq2_reference is not None and not torch.allclose(rq2_reference, rq2_logits, atol=1e-4, rtol=1e-4):
            raise ValueError("Attention capture changed logits beyond numerical tolerance.")
        if set(rq2_weights) != set(RQ2_BLOCKS):
            raise ValueError("Missing requested attention blocks.")
        rq2_measured = {block: _rq2_measure(rq2_weights[block], rq2_far) for block in RQ2_BLOCKS}
        for rq2_block, rq2_values in rq2_measured.items():
            rq2_valid = np.isfinite(rq2_values["dam"])
            rq2_image_metrics.append({"image_id": rq2_id, "block": rq2_block,
                "n_valid": int(rq2_valid.sum()),
                "dam": float(rq2_values["dam"][rq2_valid].mean()) if rq2_valid.any() else np.nan,
                "uniform": float(rq2_values["uniform"][rq2_valid].mean()) if rq2_valid.any() else np.nan,
                "excluded": float(rq2_values["excluded"].mean())})
        if rq2_id in rq2_example_ids:
            rq2_mean = torch.tensor(rq2_metadata["preprocessing"]["mean"])[:, None, None]
            rq2_std = torch.tensor(rq2_metadata["preprocessing"]["std"])[:, None, None]
            rq2_crop = (rq2_tensor[0].detach().cpu() * rq2_std + rq2_mean).clamp(0, 1).permute(1, 2, 0).numpy()
            rq2_indices = [row * RQ2_GRID + col for row, col in RQ2_QUERIES]
            rq2_examples.append({"image_id": rq2_id, "crop": rq2_crop,
                "maps": {block: values["conditional"][rq2_indices].copy() for block, values in rq2_measured.items()},
                "dam": {block: values["dam"][rq2_indices].copy() for block, values in rq2_measured.items()}})
        if rq2_count % 100 == 0 or rq2_count == len(rq2_selection):
            print(f"RQ2 processed {rq2_count}/{len(rq2_selection)} images")
finally:
    del rq2_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
if rq2_count != len(rq2_selection):
    raise ValueError("Not all requested RQ2 images were processed.")

print("\nRQ2 summary: equal weight per image; queries averaged within image; percentages of OTHER-patch attention")
for rq2_block in RQ2_BLOCKS:
    rq2_rows = [row for row in rq2_image_metrics if row["block"] == rq2_block]
    rq2_valid_rows = [row for row in rq2_rows if np.isfinite(row["dam"])]
    rq2_valid_queries = sum(row["n_valid"] for row in rq2_rows)
    if not rq2_valid_rows:
        print(f"Block {rq2_block}: DAM undefined; 0 valid images. No zero substitution.")
        continue
    rq2_dam = np.array([row["dam"] for row in rq2_valid_rows])
    rq2_uniform = np.array([row["uniform"] for row in rq2_valid_rows])
    rq2_excluded = np.mean([row["excluded"] for row in rq2_rows])
    print(f"Block {rq2_block:2d}: DAM={rq2_dam.mean():.2%}; uniform={rq2_uniform.mean():.2%}; "
          f"difference={100 * (rq2_dam - rq2_uniform).mean():+.2f} pp; near={1 - rq2_dam.mean():.2%}")
    print(f"  Valid images={len(rq2_valid_rows)}/{len(rq2_rows)}, queries={rq2_valid_queries}/{len(rq2_rows) * RQ2_GRID ** 2}; "
          f"excluded CLS+self mass={rq2_excluded:.2%} of original attention")


In [ ]:
from matplotlib.patches import Rectangle, Circle
from matplotlib.ticker import PercentFormatter

for rq2_example in rq2_examples:
    rq2_fig, rq2_axes = plt.subplots(len(RQ2_QUERIES), 5, figsize=(17, 3.7 * len(RQ2_QUERIES)),
                                    squeeze=False, constrained_layout=True)
    rq2_height, rq2_width = rq2_example["crop"].shape[:2]
    for rq2_q, (rq2_row, rq2_col) in enumerate(RQ2_QUERIES):
        rq2_axrow = rq2_axes[rq2_q]
        rq2_finite = np.concatenate([rq2_example["maps"][b][rq2_q] for b in RQ2_BLOCKS])
        rq2_finite = rq2_finite[np.isfinite(rq2_finite)]
        rq2_vmax = float(rq2_finite.max()) if len(rq2_finite) else 1.0
        rq2_vmax = max(rq2_vmax, np.finfo(float).eps)
        for rq2_column, rq2_ax in enumerate(rq2_axrow):
            rq2_ax.imshow(rq2_example["crop"], extent=(0, RQ2_GRID, RQ2_GRID, 0))
            if rq2_column:
                rq2_block = RQ2_BLOCKS[rq2_column - 1]
                rq2_heat = rq2_example["maps"][rq2_block][rq2_q].reshape(RQ2_GRID, RQ2_GRID).copy()
                rq2_heat[rq2_row, rq2_col] = np.nan
                rq2_artist = rq2_ax.imshow(np.ma.masked_invalid(rq2_heat), cmap="magma", interpolation="nearest",
                    extent=(0, RQ2_GRID, RQ2_GRID, 0), vmin=0, vmax=rq2_vmax, alpha=0.8)
                rq2_ax.set_title(f"Block {rq2_block} | query DAM={rq2_example['dam'][rq2_block][rq2_q]:.1%}")
            else:
                rq2_ax.set_title(f"Official input crop | query ({rq2_row}, {rq2_col})")
            rq2_ax.add_patch(Rectangle((rq2_col, rq2_row), 1, 1, fill=False, edgecolor="yellow", linewidth=2))
            rq2_ax.add_patch(Circle((rq2_col + 0.5, rq2_row + 0.5), RQ2_FAR_DISTANCE,
                                    fill=False, edgecolor="cyan", linestyle="--", linewidth=1.2))
            rq2_ax.set(xlim=(0, RQ2_GRID), ylim=(RQ2_GRID, 0), xticks=[], yticks=[])
        rq2_fig.colorbar(rq2_artist, ax=list(rq2_axrow[1:]), shrink=0.75,
                        format=PercentFormatter(1), label="Attention per other patch (shared scale across blocks)")
    rq2_fig.suptitle(f"RQ2 | {rq2_example['image_id']} | Yellow=query; cyan radius={RQ2_FAR_DISTANCE:g}; distant=outside/on radius")
    plt.show()
print("RQ2 preview only: no results or figures saved. Review DAM against uniform before drawing conclusions.")
